# Retail Customer Purchase Intelligence — Milestone 3
**Participant:** Participant_M3_03  
**Project:** Retail Customer Purchase Intelligence  
**Milestone:** 3  
**Dataset package:** `Retail.zip`

## Objective
Predict purchase amounts, segment customers, understand purchase drivers, and support marketing/inventory decisions.

## Milestone scope
neural network regression for Purchase; compare with best ML model; NLP on Customer_Reviews.csv; document limitations.

> This is a synthetic sample submission for evaluator testing. Dataset files must be placed in the paths referenced below. Results are not pre-filled or fabricated.

## Neural network regression and comparison
Fit preprocessing on training data only, compare with a selected ML model on the same held-out test set, and use early stopping or regularization.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

demo = pd.read_csv(Path("data")/"user_demographics.csv")
purchase = pd.read_csv(Path("data")/"User_product_purchase_details.csv")
df = purchase.merge(demo, on="User_ID", how="left", validate="many_to_one")
y = pd.to_numeric(df["Purchase"], errors="coerce")
X = df.drop(columns=["Purchase", "User_ID", "Product_ID"], errors="ignore")
ok = y.notna(); X, y = X.loc[ok], y.loc[ok]
cat = X.select_dtypes(include=["object", "category"]).columns.tolist()
num = X.select_dtypes(include=np.number).columns.tolist()
prep = ColumnTransformer([
 ("num", Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
 ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore"))]),cat)
])
Xtr, Xte, ytr, yte = train_test_split(X,y,test_size=.2,random_state=42)
Xtr_t = prep.fit_transform(Xtr); Xte_t = prep.transform(Xte)
ml = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1).fit(Xtr_t,ytr)
dl = MLPRegressor(hidden_layer_sizes=(128,64), early_stopping=True, max_iter=150, random_state=42).fit(Xtr_t,ytr)
for name, model in [("RandomForest",ml),("MLP",dl)]:
 pred = model.predict(Xte_t)
 print(name, {"MAE":mean_absolute_error(yte,pred),
              "RMSE":mean_squared_error(yte,pred)**.5,"R2":r2_score(yte,pred)})

## NLP: customer review text
Use the supplied review text file. Text processing should be evaluated with a held-out set if labelled sentiment data is available; do not invent labels or accuracy.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
reviews = pd.read_csv(Path("data")/"Customer_Reviews.csv")
display(reviews.head())
print(reviews.columns.tolist())
# Exploratory text representation only until a verified text column and labelled target are identified.
text_cols = reviews.select_dtypes(include=["object"]).columns.tolist()
print("candidate text columns:", text_cols)

## Further work
Add multiple-seed or time-window stability checks where appropriate, inspect residuals by group, and document training time and limitations.